# COVID-19 Death Risk Analysis

# COVID-19 Death Rate by Age Group

## Objective

This analysis evaluates how COVID-19 death rates vary across age groups.

Understanding how mortality risk changes with age is critical for identifying vulnerable populations and interpreting the severity patterns observed in COVID-19 surveillance data.

## Metric Definition

The death rate is calculated as:

death_rate = deaths / cases with known death status

Only records with `death_yn` values of `"Yes"` or `"No"` are included in the denominator.

This approach prevents unknown or missing values from biasing the rate calculation.

## Data Preparation

To ensure accurate rate calculations:

• Cases with unknown death status are removed  
• Age groups are standardized to a consistent format  
• Records labeled `"Missing"` or `"Unknown"` for age are excluded

After cleaning, cases are grouped by age group to calculate death counts and death rates.

## Results

The analysis calculates death rates for each age group using the filtered dataset.

The resulting table and chart show how mortality risk varies across age categories in the surveillance sample.

## Interpretation

The results show clear differences in mortality risk between age groups.

Older age groups show higher death rates, reflecting the well-documented relationship between age and COVID-19 severity.

Although the dataset used here is a sample rather than the full CDC dataset, the observed pattern is consistent with widely reported epidemiological findings.

## Limitations

This analysis uses a 1 million row sample retrieved through the CDC Socrata API.

Because the sample is extracted through API pagination rather than random sampling, some age groups may be underrepresented.

As a result, the calculated death rates should be interpreted as exploratory estimates rather than population-level statistics.

In [ ]:
import pandas as pd

df_clean = pd.read_parquet('../data/processed/cdc_clean_1M.parquet')

In [ ]:
df_known_deaths = df_clean[df_clean['death_yn'].isin(['Yes', 'No'])].copy()

df_known_deaths['age_group'] = (
    df_known_deaths['age_group']
    .str.replace(' Years', '', case = False, regex = False)
    .str.replace(' ', '', regex = False)
)

df_known_deaths = df_known_deaths[~df_known_deaths['age_group'].isin(['Missing', 'Unknown'])]

df_known_deaths.head()

In [ ]:
age_order = [
    "10-19",
    "20-29",
    "30-39",
    "40-49",
    "50-59",
    "60-69",
    "70-79"
]

death_cases = (
    df_known_deaths[df_known_deaths['death_yn'] == 'Yes'].
    groupby('age_group').
    size()
)

known_cases = (
    df_known_deaths.
    groupby('age_group').
    size()
)


In [ ]:
death_summary = pd.DataFrame({
    'known_cases': known_cases,
    'death_cases': death_cases
})

death_summary['death_rate'] = (
    (death_summary['death_cases']/
     death_summary['known_cases'])*100
)

death_summary = death_summary.reindex(age_order)
death_summary = death_summary.fillna(0)
death_summary['death_rate'] = death_summary['death_rate'].round(2)

In [ ]:
death_summary

In [ ]:
death_summary.to_csv('../outputs/death_by_age.csv')

In [ ]:
import matplotlib.pyplot as plt

plot_df = death_summary[death_summary["known_cases"] > 0]

plt.figure(figsize=(10,6))

plt.bar(
    plot_df.index,
    plot_df["death_rate"]
)

plt.xlabel("Age Group")
plt.ylabel("Death Rate (%)")
plt.ylim(0,3)

plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.title("Death Rate by Age Group (CDC COVID-19 Case Surveillance Sample)")

plt.tight_layout()

plt.savefig("../outputs/death_by_age.png")
plt.show()